In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V1 D0.1 artifact-only selection

Run once, top to bottom. This CPU-only handoff reads the frozen D0 public artifact and writes a create-only D0.1 receipt. It does not rerun D0 and retains `science_denominator=0`; no route, method, detector, keyed-anchor, or scientific conclusion follows.


In [ ]:
import json, os, pathlib, subprocess, sys
from datetime import datetime, timezone

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
SOURCE_ARTIFACT_EXACT = '4732211beefbeface95cb842c117b9719e362f1a'
D01_RUNNER_EXACT = 'ccfb7bcefbb18f9812a4e800bbea18b91b031ebb'
SOURCE_RUN_ID = 'geometry-v1-qk-d0-4732211beefb'
SOURCE_PROTOCOL = 'geometry-v1-qk-d0-all-layer-discovery-v1'
SOURCE_PLAN_DIGEST = '96e1e5ae6fb8ae66a545b1b10d6c896176989272c81ef1fd737184dcdfaea7b8'
RUNNER_PATH = 'experiments/run_geometry_v1_qk_d01_artifact_selection_operational.py'
SOURCE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/D0/Geometry-V1-QK-D0-4732211beefb-20260827T064555Z')
OUTPUT_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/D01')
SUCCESS_PREFIX = 'CEGWM_GEOMETRY_V1_QK_D01 '
FAILURE_PREFIX = 'CEGWM_GEOMETRY_V1_QK_D01_FAILURE '
MAX_CONTROL_BYTES = 1024
D01_FAILED = False
RUNNER_ATTEMPTED = False

def public_error_class(error):
    if isinstance(error, (FileExistsError, FileNotFoundError, PermissionError, OSError)):
        return 'filesystem_error'
    if isinstance(error, (ValueError, TypeError)):
        return 'validation_error'
    if isinstance(error, subprocess.SubprocessError):
        return 'subprocess_error'
    if isinstance(error, RuntimeError):
        return 'runtime_error'
    return 'unexpected_error'

def stop(stage, error):
    global D01_FAILED
    if not D01_FAILED:
        D01_FAILED = True
        print('CEGWM_GEOMETRY_V1_QK_D01_HANDOFF_FAILURE ' + json.dumps({'stage': stage, 'error_class': public_error_class(error)}, sort_keys=True, separators=(',', ':')))

def git_output(repo, *args):
    return subprocess.run(['git', *args], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()


In [ ]:
if not D01_FAILED:
    try:
        session_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        repo = pathlib.Path('/content') / ('geometry-v1-qk-d01-' + session_utc)
        if repo.exists(): raise FileExistsError('fresh checkout path exists')
        subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(repo)], check=True)
        subprocess.run(['git', 'checkout', '--detach', D01_RUNNER_EXACT], cwd=repo, check=True)
        execution_commit = git_output(repo, 'rev-parse', 'HEAD')
        checkout_clean = git_output(repo, 'status', '--porcelain') == ''
        if execution_commit != D01_RUNNER_EXACT or not checkout_clean: raise RuntimeError('runner checkout identity mismatch')
        runner_path = repo / RUNNER_PATH
        if not runner_path.is_file(): raise FileNotFoundError('D0.1 runner is unavailable at runner checkout')
        source_artifact_identity = {'execution_exact': SOURCE_ARTIFACT_EXACT, 'run_id': SOURCE_RUN_ID, 'protocol': SOURCE_PROTOCOL, 'plan_digest': SOURCE_PLAN_DIGEST, 'path': str(SOURCE_ROOT)}
        print('CEGWM_GEOMETRY_V1_QK_D01_CHECKOUT ' + json.dumps({'runner_execution_identity': {'commit': execution_commit, 'clean': checkout_clean}, 'source_artifact_identity': source_artifact_identity}, sort_keys=True))
    except BaseException as error:
        stop('checkout', error)


In [ ]:
control_read = control_write = None
if not D01_FAILED:
    try:
        if RUNNER_ATTEMPTED: raise RuntimeError('runner already attempted')
        RUNNER_ATTEMPTED = True
        if not SOURCE_ROOT.is_dir(): raise FileNotFoundError('fixed D0 source is unavailable')
        run_dir = OUTPUT_ROOT / ('Geometry-V1-QK-D01-' + execution_commit[:12] + '-' + session_utc)
        if run_dir.exists(): raise FileExistsError('create-only Drive run directory exists')
        control_read, control_write = os.pipe()
        command = [sys.executable, str(runner_path), '--repo-root', str(repo), '--expected-exact', execution_commit, '--source-root', str(SOURCE_ROOT), '--output-root', str(run_dir), '--control-fd', str(control_write)]
        process = subprocess.Popen(command, cwd=repo, pass_fds=(control_write,), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        os.close(control_write); control_write = None
        runner_rc = process.wait(timeout=7200)
        line = os.read(control_read, MAX_CONTROL_BYTES + 1)
        if len(line) > MAX_CONTROL_BYTES or not line.endswith(b'\n'): raise RuntimeError('invalid compact control receipt')
        text = line.decode('utf-8', 'strict').strip()
        if text.startswith(SUCCESS_PREFIX): receipt_status, receipt = 'success', json.loads(text[len(SUCCESS_PREFIX):])
        elif text.startswith(FAILURE_PREFIX): receipt_status, receipt = 'failure', json.loads(text[len(FAILURE_PREFIX):])
        else: raise RuntimeError('unexpected control prefix')
        if receipt.get('run_id') != 'geometry-v1-qk-d01-' + execution_commit[:12]: raise RuntimeError('receipt identity mismatch')
        terminal = {'run_id': receipt.get('run_id'), 'runner_execution_identity': {'commit': execution_commit}, 'source_artifact_identity': source_artifact_identity, 'd01_status': receipt.get('d01_status'), 'artifact_status': receipt.get('artifact_status'), 'failure_point': receipt.get('failure_point'), 'error_class': receipt.get('error_class'), 'selected_layer_paths': receipt.get('selected_layer_paths', []), 'science_denominator': 0, 'drive_directory': str(run_dir), 'runner_rc': runner_rc}
        print('CEGWM_GEOMETRY_V1_QK_D01_TERMINAL ' + json.dumps(terminal, sort_keys=True, separators=(',', ':')))
        if receipt_status != 'success': raise RuntimeError('runner reported failure')
    except BaseException as error:
        stop('runner', error)
    finally:
        for fd in (control_read, control_write):
            if fd is not None: os.close(fd)


The notebook reads only a compact control receipt. It does not display source ZIP contents, retry, fall back, switch layers, tune, or choose per sample. Any failure stops and leaves bounded create-only output for later audit.
